# Reproducible direct vs suffix-SMC validation

This notebook compares the suffix-SMC mixture with IID direct sampling when both use the same model, prompt, generation settings, and public full-sequence ARI decoding convention.

Run the cells from top to bottom. **Phase A:** direct flag only. **Phase B:** SMC flag only. **Phase C:** both flags false to reload and analyze. Only the direct ARI archive, its summary, and completed SMC component archives persist on Drive; tables and plots remain inline.

In [ ]:
# 1. Connect Drive, clone/update the repository, and install dependencies.
from google.colab import drive
drive.mount('/content/drive')

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/FrancescoVinciguerra/rea_of_llms_supplementary_code-main.git'
REPO_ROOT = Path('/content/rea_of_llms_supplementary_code-main')

if (REPO_ROOT / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only', 'origin', 'main'], check=True)
elif REPO_ROOT.exists():
    raise RuntimeError(f'{REPO_ROOT} exists but is not a Git repository.')
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)

os.chdir(REPO_ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
print('Ready:', REPO_ROOT)

In [ ]:
# 2. EDIT PARAMETERS HERE.
# Phase A: direct=True, SMC=False. Phase B: direct=False, SMC=True.
# Phase C (analysis only): both False.
RUN_OFFICIAL_PUBLIC_DIRECT = False
RUN_OFFICIAL_PUBLIC_SMC = False
RUN_DRIVE_AUDIT = False  # Optional metadata-only inspection; generates nothing.
ARI_CONVENTION = 'official_public'

# Change this one path if your data lives in a different Drive folder.
OUTPUT_ROOT = Path('/content/drive/MyDrive/rea_official_public_validation')

COMPONENTS = [
    {'cap': 6,  'lambda': -0.5},
    {'cap': 6,  'lambda': -1.0},
    {'cap': 6,  'lambda': -1.5},
    {'cap': 7,  'lambda': -1.0},
    {'cap': 7,  'lambda': -1.5},
    {'cap': 7,  'lambda': -2.0},
    {'cap': 8,  'lambda': -1.5},
    {'cap': 8,  'lambda': -2.0},
    {'cap': 8,  'lambda': -2.5},
]
SEED_GROUPS = [500]
TAIL_THRESHOLDS = [6, 7, 8]
TAIL_THRESHOLDS_CSV = ','.join(map(str, TAIL_THRESHOLDS))
NUM_PARTICLES = 256
MCMC_STEPS_PER_LEVEL = 10
LAMBDA_STEP = 0.25
ESS_THRESHOLD = 0.5
MAX_NEW_TOKENS = 100
SUFFIX_SIZES = '1,2,4,8,16,32,64,100'
SUFFIX_RHOS = '0.20,0.18,0.16,0.14,0.12,0.10,0.06,0.04'
BATCH_SIZE = 32
DEVICE = 'cuda'

OFFICIAL_PUBLIC_DIRECT_NUM_SAMPLES = 200_000
OFFICIAL_PUBLIC_DIRECT_BATCH_SIZE = 256
OFFICIAL_PUBLIC_DIRECT_SEED = 91_001

MODEL_NAME = 'roneneldan/TinyStories-8M'
TOKENIZER_NAME = 'EleutherAI/gpt-neo-125M'
PROMPT = 'Once upon a time, in a big forest, there lived a rhinoc'
TEMPERATURE = 1.0

assert ARI_CONVENTION == 'official_public'
assert TAIL_THRESHOLDS == [6, 7, 8] and TEMPERATURE == 1.0
print(f'Phase A - generate direct: {RUN_OFFICIAL_PUBLIC_DIRECT}')
print(f'Phase B - generate SMC:    {RUN_OFFICIAL_PUBLIC_SMC}')
print(f'Phase C - analysis only:   {not RUN_OFFICIAL_PUBLIC_DIRECT and not RUN_OFFICIAL_PUBLIC_SMC}')
print(f'Planned components:  {len(COMPONENTS) * len(SEED_GROUPS)}')
print(f'Data folder:         {OUTPUT_ROOT}')

In [ ]:
# 3. Optional read-only audit of existing Drive artifacts. This cell never writes files.
import pandas as pd
from IPython.display import display
from rea_llms.data_reuse_audit import REUSE_DIMENSIONS, audit_run_directory

if RUN_DRIVE_AUDIT:
    audit_rows = audit_run_directory(
        OUTPUT_ROOT, target_ari_convention=ARI_CONVENTION,
    )
    if not audit_rows:
        print(f'No recognized run artifacts found under {OUTPUT_ROOT}')
    else:
        audit_df = pd.DataFrame(audit_rows)
        inventory_columns = [
            'artifact_type', 'algorithm', 'ari_convention', 'prompt_sha256',
            'model_name', 'tokenizer_name', 'max_new_tokens', 'ari_cap',
            'lambda_final', 'num_particles', 'mcmc_steps_per_level', 'seed',
            'has_raw_ari', 'has_final_smc_weights', 'has_log_z',
            'has_token_ids', 'runtime_seconds', 'generated_token_count',
            'proposal_count', *REUSE_DIMENSIONS, 'path',
        ]
        display(audit_df.reindex(columns=inventory_columns))
        failed = []
        for _, row in audit_df.iterrows():
            reasons = [row.get(f'reason_{key}') for key in REUSE_DIMENSIONS if not row.get(key, False)]
            failed.append('; '.join(reason for reason in reasons if isinstance(reason, str)))
        limitations = audit_df[['path']].copy()
        limitations['limitations'] = failed
        display(limitations)
        print(f'Read-only audit complete: {len(audit_df)} artifacts; no samples generated.')
else:
    print('Drive audit disabled. Set RUN_DRIVE_AUDIT=True to inspect existing artifacts.')


In [ ]:
# 4. Phase A: generate once, or load, the official-public IID direct reference.
import json
import torch

print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if DEVICE == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('DEVICE=cuda but this Colab runtime has no GPU.')

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
DIRECT_ROOT = OUTPUT_ROOT / 'official_public_direct_reference'
if RUN_OFFICIAL_PUBLIC_DIRECT:
    command = [
        sys.executable, '-m', 'rea_llms.current_direct_reference',
        '--model-name', MODEL_NAME, '--tokenizer-name', TOKENIZER_NAME,
        '--prompt', PROMPT, '--device', DEVICE,
        '--max-new-tokens', str(MAX_NEW_TOKENS),
        '--num-samples', str(OFFICIAL_PUBLIC_DIRECT_NUM_SAMPLES),
        '--batch-size', str(OFFICIAL_PUBLIC_DIRECT_BATCH_SIZE),
        '--seed', str(OFFICIAL_PUBLIC_DIRECT_SEED),
        '--thresholds', TAIL_THRESHOLDS_CSV,
        '--ari-convention', ARI_CONVENTION,
        '--output-dir', str(DIRECT_ROOT),
        '--skip-completed',
    ]
    subprocess.run(command, check=True)
direct_summary_path = DIRECT_ROOT / 'summary.json'
if not direct_summary_path.is_file():
    raise FileNotFoundError('Official-public direct cache missing. Run Phase A once.')
direct_summary = json.loads(direct_summary_path.read_text())
metadata = direct_summary['metadata']
assert metadata['ari_convention'] == ARI_CONVENTION
assert metadata['prompt_sha256'] == 'b4b7f60343cd3d1e24d8f2e312810afdaaa28034b9a194c66865b47483f70f59'
assert metadata['model_name'] == MODEL_NAME and metadata['tokenizer_name'] == TOKENIZER_NAME
assert metadata['max_new_tokens'] == MAX_NEW_TOKENS
assert 'decode the full prompt-token plus completion-token sequence' in metadata['decoding_convention']
assert direct_summary['runtime_seconds'] is not None
assert direct_summary['generated_token_count'] == OFFICIAL_PUBLIC_DIRECT_NUM_SAMPLES * MAX_NEW_TOKENS
print(f"Official-public direct ready: N={direct_summary['statistics']['count']:,}; runtime={direct_summary['runtime_seconds']/60:.1f} min")

In [ ]:
# 5. Phase B: generate missing official-public SMC components, or load exact caches.
# This cell does not write reports, tables, or plots to Drive.
from suffix_direct_mh_smc_experiment.run_tail_validation import parse_args, run_validation

arguments = [
    '--model-name', MODEL_NAME, '--tokenizer-name', TOKENIZER_NAME,
    '--prompt', PROMPT, '--device', DEVICE,
    '--components-json', json.dumps(COMPONENTS),
    '--seed-groups', ','.join(map(str, SEED_GROUPS)),
    '--thresholds', TAIL_THRESHOLDS_CSV, '--ari-convention', ARI_CONVENTION,
    '--lambda-step', str(LAMBDA_STEP),
    '--num-particles', str(NUM_PARTICLES), '--max-new-tokens', str(MAX_NEW_TOKENS),
    '--mcmc-steps-per-level', str(MCMC_STEPS_PER_LEVEL),
    '--ess-threshold', str(ESS_THRESHOLD), '--suffix-sizes', SUFFIX_SIZES,
    '--suffix-rhos', SUFFIX_RHOS, '--batch-size', str(BATCH_SIZE),
    '--output-root', str(OUTPUT_ROOT),
    '--direct-summary', str(direct_summary_path), '--skip-completed-runs',
]
if not RUN_OFFICIAL_PUBLIC_SMC:
    arguments.append('--audit-only')

validation = run_validation(
    parse_args(arguments),
    persist_analysis=False,
    fast_cache=True,
)

status = validation.component_status.groupby(['seed_group', 'status']).size().unstack(fill_value=0)
print('Component cache:')
print(status.to_string())
print('Complete seed groups used:', sorted(validation.complete_grouped))
if not validation.complete_grouped:
    print('SMC mixture unavailable: complete Phase B before the final comparison.')
else:
    assert sorted(validation.complete_grouped) == SEED_GROUPS
    assert len(validation.specs) == len(COMPONENTS)
    print('All official-public SMC components are complete and exactly cache-compatible.')

In [ ]:
# 6. Direct-vs-SMC comparison plus component diagnostics.
import math
import numpy as np
import pandas as pd
from IPython.display import display

direct_stats = direct_summary['statistics']
direct_tails = {float(row['threshold']): row for row in direct_stats['tails']}
aggregate = validation.comparison.query("row_type == 'aggregate'").copy()
smc_rows = {float(row['threshold']): row for _, row in aggregate.iterrows()}

result_rows = []
for threshold in map(float, TAIL_THRESHOLDS):
    direct = direct_tails[threshold]
    p_direct = float(direct['empirical_probability'])
    smc = smc_rows.get(threshold)
    p_smc = math.nan if smc is None else float(smc['mixture_probability'])
    ratio = p_smc / p_direct if smc is not None and p_direct > 0 else math.nan
    result_rows.append({
        'ARI threshold': threshold,
        'direct events': int(direct['count']),
        'direct probability': p_direct,
        'direct Wilson 95%': f"[{direct['wilson_95_lower']:.7g}, {direct['wilson_95_upper']:.7g}]",
        'SMC-mixture probability': p_smc,
        'SMC / direct': ratio,
        'relative difference': ratio - 1.0 if math.isfinite(ratio) else math.nan,
        'mixture ESS/N': math.nan if smc is None else float(smc['weight_ess_over_n']),
        'direct runtime (s)': float(direct_summary['runtime_seconds']),
        'SMC runtime (s)': math.nan if smc is None else float(smc['runtime_seconds']),
        'direct generated tokens': int(direct_summary['generated_token_count']),
        'SMC generated tokens': math.nan if smc is None else smc['generated_token_count'],
    })

results = pd.DataFrame(result_rows)
display(results.style.format({
    'ARI threshold': '{:.0f}', 'direct events': '{:,}',
    'direct probability': '{:.7g}',
    'SMC-mixture probability': '{:.7g}', 'SMC / direct': '{:.3f}',
    'relative difference': '{:+.1%}', 'mixture ESS/N': '{:.3f}',
    'direct runtime (s)': '{:,.1f}', 'SMC runtime (s)': '{:,.1f}',
    'direct generated tokens': '{:,.0f}', 'SMC generated tokens': '{:,.0f}',
}, na_rep='not available'))

if not validation.component_diagnostics.empty:
    diagnostics = validation.component_diagnostics[[
        'cap', 'lambda_final', 'threshold', 'component_probability',
        'corrected_ess_over_n', 'raw_event_count', 'log_z_smc',
        'log_z_reverse', 'delta_log_z_smc_minus_reverse',
    ]]
    print('Component diagnostics:')
    display(diagnostics.style.format({
        'cap': '{:g}', 'lambda_final': '{:g}', 'threshold': '{:g}',
        'component_probability': '{:.7g}', 'corrected_ess_over_n': '{:.3f}',
        'raw_event_count': '{:,}', 'log_z_smc': '{:.5f}',
        'log_z_reverse': '{:.5f}', 'delta_log_z_smc_minus_reverse': '{:+.5f}',
    }))

smc_available = bool(results['SMC-mixture probability'].notna().all())
print('Interpretation:')
if not smc_available:
    print('- Direct reference is available; complete Phase B to estimate the SMC mixture.')
else:
    differences = results['relative difference'].to_numpy(float)
    print('- Relative SMC/direct differences:', ', '.join(f'{value:+.1%}' for value in differences))
    print(f"- Corrected mixture ESS/N: {results['mixture ESS/N'].iloc[0]:.3f}")
print('- One seed group does not estimate between-seed variance.')

In [ ]:
# 7. Exactly two inline plots; no PNG or other analysis files are written.
import matplotlib.pyplot as plt

x = results['ARI threshold'].to_numpy(float)
direct_p = results['direct probability'].to_numpy(float)
smc_p = results['SMC-mixture probability'].to_numpy(float)
ratios = results['SMC / direct'].to_numpy(float)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
ax = axes[0]
ax.plot(x, direct_p, marker='s', label='IID direct', color='#2C7BB6')
if np.isfinite(smc_p).all():
    ax.plot(x, smc_p, marker='^', label='SMC mixture', color='#D95F02')
ax.set_yscale('log'); ax.set_xticks(x); ax.set_xlabel('raw ARI threshold')
ax.set_ylabel('tail probability'); ax.set_title('Tail probabilities')
ax.grid(alpha=.25, which='both'); ax.legend()

ax = axes[1]
if np.isfinite(ratios).all():
    ax.bar(x.astype(int).astype(str), ratios, color='#D95F02')
    ax.axhline(1.0, color='black', linewidth=1)
else:
    ax.text(0.5, 0.5, 'Complete Phase B to show SMC/direct ratios', ha='center', va='center', transform=ax.transAxes)
ax.set_xlabel('raw ARI threshold'); ax.set_ylabel('SMC / IID direct')
ax.set_title('SMC/direct ratio'); ax.grid(axis='y', alpha=.25)
fig.tight_layout()
plt.show()